# Provider Files Exploration

Exploring provider order files to find join keys with marketplace data

In [ ]:
import pandas as pd
import numpy as np
import glob

# Load all provider files
provider_files = glob.glob('td68ff*.csv')
print(f"Found {len(provider_files)} provider files:")
for f in provider_files:
    print(f"  - {f}")

# Load all provider data
provider_dfs = []
for file in provider_files:
    df = pd.read_csv(file, encoding='utf-8')
    df['source_file'] = file
    provider_dfs.append(df)
    print(f"\n{file}: {len(df)} rows, {len(df.columns)} columns")

# Combine all provider data
df_provider = pd.concat(provider_dfs, ignore_index=True)
print(f"\nTotal provider orders: {len(df_provider)}")

In [ ]:
# Explore structure
print("Column names:")
print(df_provider.columns.tolist())
print("\nFirst few rows:")
df_provider.head(10)

In [ ]:
# Check 'Riferimento interno' (Internal Reference) - this might contain marketplace order numbers
print("Riferimento interno samples (non-empty):")
non_empty_ref = df_provider[df_provider['Riferimento interno'].notna() & (df_provider['Riferimento interno'] != '')]
print(f"\nTotal non-empty references: {len(non_empty_ref)} out of {len(df_provider)}")
print("\nSamples:")
print(non_empty_ref[['Id. Ordine', 'Data', 'Riferimento interno', 'Paese', 'Stato dell\'ordine']].head(20))

In [ ]:
# Check Fattura (Invoice) field - might contain order references
print("Fattura (Invoice) samples:")
non_empty_invoice = df_provider[df_provider['Fattura'].notna() & (df_provider['Fattura'] != '')]
print(f"\nTotal with invoice: {len(non_empty_invoice)} out of {len(df_provider)}")
print("\nSamples:")
print(non_empty_invoice[['Id. Ordine', 'Data', 'Fattura', 'Paese', 'Stato dell\'ordine', 'Totale']].head(20))

In [ ]:
# Load a sample of marketplace data to compare
# Let's use German data as example
gmu_file = glob.glob('report_booking_gmu_de_*.csv')[0]
df_gmu = pd.read_csv(gmu_file, sep=';', decimal=',', thousands='.', encoding='utf-8')

print("Sample marketplace order numbers from GMU:")
sample_orders = df_gmu[df_gmu['order_number'].notna()]['order_number'].head(10).tolist()
print(sample_orders)

print("\nSample marketplace customer names from GMU:")
print(df_gmu[df_gmu['order_number'].notna()][['order_number', 'shipping.first_name', 'shipping.last_name', 'shipping.city']].head(10))

In [ ]:
# Try to find matching names between provider and marketplace
# Create full name in provider data
df_provider['full_name'] = df_provider['Nome'].fillna('') + ' ' + df_provider['Cognome'].fillna('')
df_provider['full_name'] = df_provider['full_name'].str.strip().str.lower()

# Create full name in GMU data
df_gmu['full_name'] = df_gmu['shipping.first_name'].fillna('') + ' ' + df_gmu['shipping.last_name'].fillna('')
df_gmu['full_name'] = df_gmu['full_name'].str.strip().str.lower()

# Convert dates
df_provider['Data'] = pd.to_datetime(df_provider['Data'])
df_gmu['order_date'] = pd.to_datetime(df_gmu['order_date'])

print("Provider data date range:")
print(f"  From: {df_provider['Data'].min()}")
print(f"  To: {df_provider['Data'].max()}")

print("\nGMU data date range:")
print(f"  From: {df_gmu['order_date'].min()}")
print(f"  To: {df_gmu['order_date'].max()}")

In [ ]:
# Check if we can match by: name + city + date (within 1-2 days)
# Sample attempt with a few GMU orders
print("Attempting to match GMU orders with provider orders...\\n")

sample_gmu = df_gmu[df_gmu['order_number'].notna()].head(5)

for idx, gmu_row in sample_gmu.iterrows():
    print(f"\\nGMU Order: {gmu_row['order_number']}")
    print(f"  Name: {gmu_row['full_name']}")
    print(f"  City: {gmu_row['shipping.city']}")
    print(f"  Date: {gmu_row['order_date']}")
    print(f"  Price: {gmu_row['price_gross']}")
    
    # Try to find in provider
    matches = df_provider[
        (df_provider['full_name'] == gmu_row['full_name']) &
        (df_provider['country_code'] == 'DE') &
        (abs((df_provider['Data'] - gmu_row['order_date']).dt.days) <= 2)
    ]
    
    if len(matches) > 0:
        print(f"  ✓ FOUND {len(matches)} potential match(es) in provider:")
        status_col = "Stato dell'ordine"
        print(matches[['Id. Ordine', 'Data', 'Totale', status_col, 'Fattura']].to_string())
    else:
        print("  ✗ No match found")

In [ ]:
# Check if we can match by: name + city + date (within 1-2 days)
# Sample attempt with a few GMU orders
print("Attempting to match GMU orders with provider orders...\n")

sample_gmu = df_gmu[df_gmu['order_number'].notna()].head(5)

for idx, gmu_row in sample_gmu.iterrows():
    print(f"\nGMU Order: {gmu_row['order_number']}")
    print(f"  Name: {gmu_row['full_name']}")
    print(f"  City: {gmu_row['shipping.city']}")
    print(f"  Date: {gmu_row['order_date']}")
    print(f"  Price: {gmu_row['price_gross']}")
    
    # Try to find in provider
    matches = df_provider[
        (df_provider['full_name'] == gmu_row['full_name']) &
        (df_provider['country_code'] == 'DE') &
        (abs((df_provider['Data'] - gmu_row['order_date']).dt.days) <= 2)
    ]
    
    if len(matches) > 0:
        print(f"  ✓ FOUND {len(matches)} potential match(es) in provider:")
        print(matches[['Id. Ordine', 'Data', 'Totale', 'Stato dell\'ordine', 'Fattura']].to_string())
    else:
        print("  ✗ No match found")

In [ ]:
# Check provider order statuses
print("Provider order statuses:")
status_col = "Stato dell'ordine"
print(df_provider[status_col].value_counts())

print("\nProvider data summary:")
print(f"Total orders: {len(df_provider)}")
print(f"Total value: {df_provider['Totale'].sum():.2f}")
delivered_count = len(df_provider[df_provider[status_col] == 'DELIVERED'])
processing_count = len(df_provider[df_provider[status_col] == 'PROCESSING'])
cancelled_count = len(df_provider[df_provider[status_col] == 'CANCELLED'])
print(f"Delivered: {delivered_count}")
print(f"Processing: {processing_count}")
print(f"Cancelled: {cancelled_count}")